### imports, constants, paths

In [7]:
import json, re
from pathlib import Path
import numpy as np, zarr, torch
from skimage.feature import peak_local_max
from detection.unet3d import UNet3D
import networkx as nx, pandas as pd, time
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# ---------- model ----------
VERSION = "v3"
EPOCH = 10
CKPT = f"detection/models/unet_{VERSION}_e{EPOCH}.pt"
NORM_PATH = f"detection/models/norm_{VERSION}.json"      # must be the norm file this model was trained with
UNET_C = 16

# ---------- detection params ----------
PEAK_THR = 0.3        # lenient: saved candidate peaks (filter later with conf)
DIST = 6              # xy half-window used for the peak footprint
TRACK_THR = 0.6       # the "strict" threshold, applied later via conf >= TRACK_THR (not used in inference)
PHYS = np.array([1.625, 0.40625, 0.40625])     # um per voxel (z, y, x)

# ---------- data / paths ----------
VIDEOS = ["44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"]
ZARR_FMT = "test_processed/{}.zarr/0"               # (T, 64, 256, 256), preprocessed like training
GEFF_FMT = "train/{}.geff"                      # <- adjust
OUT_TAG = f"{VERSION}_e{EPOCH}"
PTS_FMT = "pred_centroids/{}_" + OUT_TAG + "_pts.npy"          # (n, 5): t, z, y, x, conf
HM_FMT  = "pred_heatmaps/{}_" + OUT_TAG + "_hm.npy"    # (T, 64, 256, 256) float16
OVERWRITE_PREDS = True
OVERWRITE_TRACKS = True

for d in ("pred_centroids", "pred_heatmaps", "pred_tracks"):
    Path(d).mkdir(exist_ok=True)

# peak footprint: z half-window scaled for 4x anisotropy (same as the sweeps)
dz = max(1, round(DIST / 4))
FP = np.ones((2 * dz + 1, 2 * DIST + 1, 2 * DIST + 1), bool)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| ckpt:", CKPT, "| norm:", NORM_PATH)

device: cuda | ckpt: detection/models/unet_v3_e10.pt | norm: detection/models/norm_v3.json


### inference, saving heatmaps and points

In [ ]:
nrm = json.load(open(NORM_PATH))
model = UNet3D(UNET_C).to(device).eval()
model.load_state_dict(torch.load(CKPT, map_location=device))

for vid in VIDEOS:
    pts_path, hm_path = Path(PTS_FMT.format(vid)), Path(HM_FMT.format(vid))
    if pts_path.exists() and hm_path.exists() and not OVERWRITE_PREDS:
        print("exists, skipping", vid); continue

    z = zarr.open(ZARR_FMT.format(vid), mode="r")
    T = z.shape[0]
    hm_out = np.lib.format.open_memmap(hm_path, mode="w+", dtype=np.float16, shape=tuple(z.shape))
    
    pts = []
    for t in range(T):
        x = np.asarray(z[t]).astype(np.float32)
        x = np.clip((x - nrm["lo"]) / (nrm["hi"] - nrm["lo"] + 1e-8), 0, 1)

        with torch.no_grad():
            xb = torch.from_numpy(x)[None, None].to(device)
            with torch.autocast(device_type=device, enabled=(device == "cuda")):
                hm = model(xb)[0, 0].float().cpu().numpy()
        
        hm_out[t] = hm.astype(np.float16)
        
        c = peak_local_max(hm, footprint=FP, threshold_abs=PEAK_THR, exclude_border=False)  # (n,3) z,y,x
        
        if len(c):
            conf = hm[tuple(c.T)]
            pts.append(np.column_stack([np.full(len(c), t), c, conf]))

    hm_out.flush(); del hm_out
    pts = np.concatenate(pts).astype(np.float32) if pts else np.zeros((0, 5), np.float32)
    np.save(pts_path, pts)
    print(vid, "| frames:", T, "| peaks:", len(pts), f"({len(pts)/T:.0f}/frame)",
          "| at conf>=TRACK_THR:", int((pts[:, 4] >= TRACK_THR).sum()))

44b6_0113de3b | frames: 100 | peaks: 27773 (278/frame) | at conf>=TRACK_THR: 27139
44b6_0b24845f | frames: 100 | peaks: 64223 (642/frame) | at conf>=TRACK_THR: 57980
6bba_05b6850b | frames: 100 | peaks: 8612 (86/frame) | at conf>=TRACK_THR: 7985
6bba_05db0fb1 | frames: 100 | peaks: 81926 (819/frame) | at conf>=TRACK_THR: 77850


### candidate graph (nodes plus skip edges)

In [ ]:
CONF_MIN = 0.5        # nodes below this are not candidates (raise to shrink the ILP)
MAX_GAP = 4           # link across up to 3 missed frames (t -> t+4)
BASE_GATE = 8.0       # um for gap 1; gate grows as sqrt(gap)
K = 5                 # nearest candidates per node per gap
GAP_PEN = 2.0         # extra edge cost per skipped frame

def load_pts(vid):
    return np.load(PTS_FMT.format(vid))   

NMS_UM = 0.0     # merge peaks closer than this (um) within a frame; 0 disables

def nms_peaks(pts, radius_um):
    if radius_um <= 0 or len(pts) == 0:
        return pts
    keep = []
    t = pts[:, 0].astype(int)
    for tt in np.unique(t):
        ix = np.where(t == tt)[0]
        ix = ix[np.argsort(-pts[ix, 4])]                  # highest confidence first
        xyz = pts[ix, 1:4] * PHYS
        tree = cKDTree(xyz); dead = np.zeros(len(ix), bool)
        for k in range(len(ix)):
            if dead[k]: continue
            keep.append(ix[k])
            for j in tree.query_ball_point(xyz[k], radius_um):
                if j != k: dead[j] = True
    return pts[np.sort(keep)]

def build_candidates(pts):
    pts = pts[pts[:, 4] >= CONF_MIN]
    pts = nms_peaks(pts, NMS_UM)
    t = pts[:, 0].astype(int); xyz = pts[:, 1:4] * PHYS
    G = nx.DiGraph()
    for i in range(len(pts)):
        G.add_node(i, t=int(t[i]), conf=float(pts[i, 4]))
    by_t = {tt: np.where(t == tt)[0] for tt in np.unique(t)}
    trees = {tt: cKDTree(xyz[ix]) for tt, ix in by_t.items()}
    for t0, A in by_t.items():
        for gap in range(1, MAX_GAP + 1):
            B = by_t.get(t0 + gap)
            if B is None:
                continue
            k = min(K, len(B))
            d, j = trees[t0 + gap].query(xyz[A], k=k, distance_upper_bound=BASE_GATE * np.sqrt(gap))
            d, j = d.reshape(len(A), k), j.reshape(len(A), k)
            for a, dr, jr in zip(A, d, j):
                for dd, jj in zip(dr, jr):
                    if np.isfinite(dd):
                        G.add_edge(int(a), int(B[jj]), cost=float(dd + GAP_PEN * (gap - 1)), gap=gap)
    return G, pts

for vid in VIDEOS:
    G, p = build_candidates(load_pts(vid))
    print(vid, "nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

44b6_0113de3b nodes: 27367 edges: 366085
44b6_0b24845f nodes: 60495 edges: 862570
6bba_05b6850b nodes: 8227 edges: 75537
6bba_05db0fb1 nodes: 79412 edges: 1264230


### ILP solve, in overlapping time windows

In [29]:
import pickle
# per-video overrides; the dense and junk-heavy videos get a smaller ILP
CFG = {
    "44b6_0113de3b": dict(CONF_MIN=0.5, K=5, MAX_GAP=4, WINDOW=25, OVERLAP=5, TIMEOUT=300),
    "44b6_0b24845f": dict(CONF_MIN=0.5, K=5, MAX_GAP=4, WINDOW=15, OVERLAP=3, TIMEOUT=900),
    "6bba_05b6850b": dict(CONF_MIN=0.5, K=5, MAX_GAP=4, WINDOW=25, OVERLAP=5, TIMEOUT=300),
    "6bba_05db0fb1": dict(CONF_MIN=0.6, K=3, MAX_GAP=3, WINDOW=10, OVERLAP=3, TIMEOUT=600),
}

def cfg(vid): 
    return CFG[vid]

def set_cfg(vid):
    g = globals()
    for k, v in cfg(vid).items():
        g[k] = v

In [30]:
import motile
from motile.constraints import MaxParents, MaxChildren
from motile.costs import NodeSelectedCost, EdgeSelectedCost, NodeAppearCost
from motile.variables import NodeSelected, EdgeSelected

# cost knobs (tune these on GT edge recall)
W_CONF, NODE_CONST = -10.0, 4.0     # node cost = -10*conf + 4  -> selecting a node is rewarded when conf > 0.4
EDGE_CONST = -6.0                   # edge cost = dist + gap penalty - 6 -> edges shorter than ~6 um are rewarded
APPEAR = 15.0                       # starting a track costs this, which suppresses isolated flickers
WINDOW, OVERLAP = 25, 5             # frames per solve / overlap between solves
TIMEOUT = 300                       # seconds per window

def solve_window(G, t_lo, t_hi):
    nodes = [n for n, d in G.nodes(data=True) if t_lo <= d["t"] < t_hi]
    sub = G.subgraph(nodes).copy()
    if sub.number_of_nodes() == 0:
        return [], []
    
    tg = motile.TrackGraph(sub, frame_attribute="t")
    s = motile.Solver(tg)
    s.add_cost(NodeSelectedCost(weight=W_CONF, attribute="conf", constant=NODE_CONST))
    s.add_cost(EdgeSelectedCost(weight=1.0, attribute="cost", constant=EDGE_CONST))
    s.add_cost(NodeAppearCost(constant=APPEAR))
    s.add_constraint(MaxParents(1)); s.add_constraint(MaxChildren(2))
    
    sol = s.solve(timeout=TIMEOUT)
    nv, ev = s.get_variables(NodeSelected), s.get_variables(EdgeSelected)
    
    sel_n = [n for n in sub.nodes if sol[nv[n]] > 0.5]
    sel_e = [e for e in sub.edges if sol[ev[e]] > 0.5]
    if len(sel_e) == 0 and len(sel_n) > 0:
        print("  WARNING: 0 edges, likely timed out")
    
    return sel_n, sel_e

def track_video(G, T):
    keep_n, keep_e = set(), set()
    start = 0

    while start < T:
        end = min(start + WINDOW, T); last = end >= T
        t0 = time.time()
        n, e = solve_window(G, start, end)
        keep_to = end if last else end - OVERLAP                 # trust edges whose source frame is before this
    
        for u, v in e:
            if last or G.nodes[u]["t"] < keep_to:
                keep_e.add((u, v)); keep_n |= {u, v}
    
        keep_n |= {x for x in n if last or G.nodes[x]["t"] < keep_to}
    
        print(f"  window {start}-{end}: {len(n)} nodes, {len(e)} edges, {time.time()-t0:.0f}s")
        start = end if last else end - OVERLAP
    
    return keep_n, keep_e

INTERP = True
def finalize(pts_f, keep_n, keep_e):
    """Build the output graph; fill skipped frames of gap edges by linear interpolation."""
    F = nx.DiGraph()
    for n in keep_n:
        F.add_node(n, t=int(pts_f[n, 0]), pos=pts_f[n, 1:4].astype(float), interp=False)
    nxt = len(pts_f)
    for u, v in keep_e:
        tu, tv = F.nodes[u]["t"], F.nodes[v]["t"]
        if tv <= tu:
            continue
        if INTERP:
            prev = u
            for k in range(1, tv - tu):
                a = k / (tv - tu)
                F.add_node(nxt, t=tu + k, pos=(1 - a) * F.nodes[u]["pos"] + a * F.nodes[v]["pos"], interp=True)
                F.add_edge(prev, nxt); prev = nxt; nxt += 1
            F.add_edge(prev, v)
        else:
            F.add_edge(u, v)    
    return F

In [31]:
tracks = {}

for vid in VIDEOS:
    out = Path(f"pred_tracks/{vid}_{OUT_TAG}_ilp.pkl")
    
    if out.exists() and not OVERWRITE_TRACKS:
        tracks[vid] = pickle.load(open(out, "rb")); print("loaded", vid); continue
    
    set_cfg(vid)
    print(vid, cfg(vid))
    
    G, pts_f = build_candidates(load_pts(vid))
    print("  candidates:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
    
    T = int(pts_f[:, 0].max()) + 1
    kn, ke = track_video(G, T)
    tracks[vid] = finalize(pts_f, kn, ke)
    
    pickle.dump(tracks[vid], open(out, "wb"))
    print(vid, "tracked nodes:", tracks[vid].number_of_nodes(), "edges:", tracks[vid].number_of_edges())

44b6_0113de3b {'CONF_MIN': 0.5, 'K': 5, 'MAX_GAP': 4, 'WINDOW': 25, 'OVERLAP': 5, 'TIMEOUT': 300}
  candidates: 27367 nodes, 366085 edges
  window 0-25: 5794 nodes, 5570 edges, 26s
  window 20-45: 6534 nodes, 6292 edges, 32s
  window 40-65: 7228 nodes, 6961 edges, 38s
  window 60-85: 7517 nodes, 7213 edges, 41s
  window 80-100: 5845 nodes, 5533 edges, 26s
44b6_0113de3b tracked nodes: 28544 edges: 28328
44b6_0b24845f {'CONF_MIN': 0.5, 'K': 5, 'MAX_GAP': 4, 'WINDOW': 15, 'OVERLAP': 3, 'TIMEOUT': 900}
  candidates: 60495 nodes, 862570 edges
  window 0-15: 8142 nodes, 7611 edges, 56s
  window 12-27: 7508 nodes, 6976 edges, 48s
  window 24-39: 7175 nodes, 6692 edges, 41s
  window 36-51: 7592 nodes, 7109 edges, 40s
  window 48-63: 8792 nodes, 8251 edges, 63s
  window 60-75: 10135 nodes, 9489 edges, 74s
  window 72-87: 10764 nodes, 10062 edges, 101s
  window 84-99: 11101 nodes, 10332 edges, 103s
  window 96-100: 2930 nodes, 2180 edges, 2s
44b6_0b24845f tracked nodes: 66014 edges: 65668
6bba_0

### evaluate node and edge recall against GT

In [33]:
import zarr
MATCH_RADIUS = 7.0     # um

def load_gt_graph(path):
    g = zarr.open(path, mode="r")
    ids = g["nodes/ids"][:]
    props = {a: g[f"nodes/props/{a}/values"][:] for a in ("t", "z", "y", "x")}
    edges = g["edges/ids"][:]
    return ids, props, edges

def edge_counts(F, gt_path):
    ids, pr, edges = load_gt_graph(gt_path)
    gt_t = pr["t"].astype(int); gt_pos = np.column_stack([pr["z"], pr["y"], pr["x"]])
    idx = {n: i for i, n in enumerate(ids)}
    f_nodes = np.array(list(F.nodes)); f_t = np.array([F.nodes[n]["t"] for n in f_nodes])
    f_pos = np.array([F.nodes[n]["pos"] for n in f_nodes])
    rmatch = {}                                    # predicted node -> gt index
    for t in np.unique(gt_t):
        gi = np.where(gt_t == t)[0]; fi = np.where(f_t == t)[0]
        if len(fi) == 0: continue
        D = cdist(gt_pos[gi] * PHYS, f_pos[fi] * PHYS)
        D[D > MATCH_RADIUS] = 1e6                  # forbid matches beyond 7 um before solving
        r, c = linear_sum_assignment(D)
        for x, y in zip(r, c):
            if D[x, y] < 1e6:
                rmatch[f_nodes[fi[y]]] = gi[x]
    gt_edges = {(idx[s], idx[d]) for s, d in edges}
    succ, pred = {}, {}
    for a, b in gt_edges:
        succ.setdefault(a, set()).add(b); pred.setdefault(b, set()).add(a)

    tp = fp = 0
    for u, v in F.edges:
        a, b = rmatch.get(u), rmatch.get(v)
        if a is not None and b is not None and (a, b) in gt_edges:
            tp += 1; continue
        # FP: target matches a GT node that has a different source, or source matches a GT node with a different target
        bad_tgt = b is not None and b in pred and (a is None or a not in pred[b])
        bad_src = a is not None and a in succ and (b is None or b not in succ[a])
        if bad_tgt or bad_src:
            fp += 1
    fn = len(gt_edges) - tp
    return dict(tp=tp, fp=fp, fn=fn, jaccard=tp / max(tp + fp + fn, 1),
                n_nodes=F.number_of_nodes(), n_interp=sum(d["interp"] for _, d in F.nodes(data=True)))

rows = {v: edge_counts(tracks[v], GEFF_FMT.format(v)) for v in VIDEOS}
res = pd.DataFrame(rows).T
tot = res[["tp", "fp", "fn"]].sum()
print(res.round(3).to_string())
print("pooled edge Jaccard:", round(tot.tp / (tot.tp + tot.fp + tot.fn), 3))

                  tp     fp     fn  jaccard  n_nodes  n_interp
44b6_0113de3b   48.0    9.0    2.0    0.814  28544.0    1235.0
44b6_0b24845f   40.0   16.0    9.0    0.615  66014.0    6109.0
6bba_05b6850b  680.0  355.0  165.0    0.567   8721.0     561.0
6bba_05db0fb1  992.0  311.0  191.0    0.664  81543.0    3973.0
pooled edge Jaccard: 0.625


In [10]:
import pickle
from pathlib import Path

TRACK_FMT = "pred_tracks/{}_" + OUT_TAG + "_ilp.pkl"      # same path the tracking cell saves to

rows = {}
for v in VIDEOS:
    p = Path(TRACK_FMT.format(v))
    if not p.exists():
        print("missing, skipping:", p); continue
    with open(p, "rb") as f:
        F = pickle.load(f)
    rows[v] = edge_counts(F, GEFF_FMT.format(v))

res = pd.DataFrame(rows).T
tot = res[["gt_nodes", "nodes_found", "gt_edges", "tp", "fp", "fn"]].sum()
display(res.round(3))
print(f"pooled: node recall {tot.nodes_found / tot.gt_nodes:.3f} | "
      f"edge recall {tot.tp / tot.gt_edges:.3f} | "
      f"edge Jaccard {tot.tp / (tot.tp + tot.fp + tot.fn):.3f}")

,gt_nodes,nodes_found,gt_edges,tp,fp,fn,node_recall,edge_recall,jaccard,n_nodes,n_interp
44b6_0113de3b,52.0,52.0,50.0,48.0,7.0,2.0,1.000,0.960,0.842,27764.0,1195.0
44b6_0b24845f,51.0,49.0,49.0,41.0,13.0,8.0,0.961,0.837,0.661,63753.0,5860.0
6bba_05b6850b,861.0,847.0,845.0,692.0,324.0,153.0,0.984,0.819,0.592,8450.0,545.0
6bba_05db0fb1,1229.0,1174.0,1183.0,1010.0,264.0,173.0,0.955,0.854,0.698,79023.0,3746.0


pooled: node recall 0.968 | edge recall 0.842 | edge Jaccard 0.655
